In [22]:
import requests
import pandas as pd

In [23]:
def fetch_technical_indicators(api_key, symbol="AAPL", interval="daily", time_period=14, series_type="close", matype=1):
    """
    Fetches PPO, OBV, ATR, and ADX data from Alpha Vantage
    and merges them into a single, chronologically sorted DataFrame.
    """
    # Base URL for all requests
    base_url = f"https://www.alphavantage.co/query?symbol={symbol}&interval={interval}&apikey={api_key}&function="

    # Dictionary of indicator endpoints (VWAP removed)
    endpoints = {
        'PPO': f"{base_url}PPO&series_type={series_type}&matype={matype}",
        'OBV': f"{base_url}OBV",
        'ATR': f"{base_url}ATR&time_period={time_period}",
        'ADX': f"{base_url}ADX&time_period={time_period}"
    }

    dataframes = []

    # Loop through each indicator and fetch the data
    for indicator, url in endpoints.items():
        print(f"Fetching {indicator} data for {symbol}...")
        response = requests.get(url).json()

        # Basic Error Checking
        if "Error Message" in response:
            print(f"API Error for {indicator}: Please check your parameters.")
            continue
        if "Information" in response:
            print(f"API Rate Limit Reached for {indicator}.")
            continue

        try:
            # The actual data is typically stored in the second key of the JSON response
            data_key = list(response.keys())[1]

            # Parse Data into a DataFrame
            df = pd.DataFrame.from_dict(response[data_key], orient='index')
            df.columns = [indicator]
            df[indicator] = df[indicator].astype(float)

            dataframes.append(df)

        except Exception as e:
            print(f"Error parsing data for {indicator}: {e}")

    # If no data was fetched successfully, exit the function
    if not dataframes:
        print("No data was successfully fetched.")
        return None

    # Merge all individual DataFrames into one
    df_combined = dataframes[0]
    for df in dataframes[1:]:
        df_combined = df_combined.join(df, how='outer')

    # Convert the index to datetime objects and sort chronologically (oldest to newest)
    df_combined.index = pd.to_datetime(df_combined.index)
    df_combined = df_combined.sort_index()

    return df_combined

In [24]:
MY_API_KEY = "AMHN58QZ560HN06D"
aapl_df = fetch_technical_indicators(MY_API_KEY)

Fetching PPO data for AAPL...
Fetching OBV data for AAPL...
API Rate Limit Reached. Free tiers are limited to 25 requests/day.
